In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import polars as pl
import numpy as np
import torch
from tqdm import tqdm

from anngeno import AnnGeno
from scripts import get_burdens, get_correlations

import multiprocessing

device = "cuda" if torch.cuda.is_available() else "cpu"
num_cores = multiprocessing.cpu_count()
print(device, num_cores)

## Read Anngeno file

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag


## Read gene-trait associations

In [ ]:
gt = pl.read_parquet("/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq")[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])
gt

### Subset associations to genes and phenoptypes available in the small Anngeno

In [ ]:
subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag.phenotypes.columns)
    )

subsest_gt

### Small subset of associations to sanity check

In [ ]:
subsest_gt.filter(pl.col('gene_symbol').is_in(["CETP", "LCAT"])).write_parquet("/home/dnanexus/subset_genes.pq")

## Profile max and top2 burdens

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list

In [ ]:
associations_df_path = '/home/dnanexus/subset_genes.pq'
associations_df = pl.read_parquet(associations_df_path)
gene_list = list(associations_df['gene_id'].unique())

In [ ]:
gene_list

In [ ]:
maf = 0.001
variants_to_keep_df = ag.annotations.filter((pl.col('AF_ukb') < maf))
ag.subset_variants(set(variants_to_keep_df.select(pl.col("id")).collect()['id']))
regions_dict = ag.get_many_regions(gene_list)


In [ ]:
%%time

gene_id = 'ENSG00000130164'

region_genotypes = regions_dict[gene_id]['genotypes']
region_annotations = regions_dict[gene_id]['annotations']
annotation_list = all_annotation_list
max_burden = True

no_variant_mask = region_genotypes.sum(axis = 0) == 0

var_scores = region_annotations[annotation_list].fill_nan(0).to_numpy().astype(np.float32).transpose()  

In [ ]:
%%time

# Numba chunked

from numba import njit, prange

@njit(parallel=True)
def compute_numba_top2(score_vec, region_genotypes, chunk_size):
    n_variants, n_samples = region_genotypes.shape
    n_chunks = (n_samples + chunk_size - 1) // chunk_size

    max_vals = np.empty(n_samples, dtype=np.float32)
    top2_sums = np.empty(n_samples, dtype=np.float32)

    for c in prange(n_chunks):
        start = c * chunk_size
        end = min(start + chunk_size, n_samples)
        for s in range(start, end):
            burden = np.abs(score_vec * region_genotypes[:, s])
            if len(burden) >= 2:
                top2 = np.partition(burden, -2)[-2:]
                max_vals[s] = top2.max()
                top2_sums[s] = top2.sum()
            elif len(burden) == 1:
                max_vals[s] = burden[0]
                top2_sums[s] = burden[0]
            else:
                max_vals[s] = 0.0
                top2_sums[s] = 0.0

    return max_vals, top2_sums

for a in tqdm(range(var_scores.shape[0])):
    score_vec = var_scores[a, :]
    max_vals, top2_sum = compute_numba_top2(score_vec, region_genotypes, chunk_size=20000)
    gis_max_list.append(max_vals)
    gis_top2_sum_list.append(top2_sum)


gis_max = np.stack(gis_max_list, axis=0).T
gis_top2 = np.stack(gis_top2_sum_list, axis=0).T

# Handle no-variant samples
gis_max[no_variant_mask, :] = np.nan
gis_top2[no_variant_mask, :] = np.nan

In [ ]:
%%time

gene_id = 'ENSG00000130164'

region_genotypes = regions_dict[gene_id]['genotypes']
region_annotations = regions_dict[gene_id]['annotations']
annotation_list = all_annotation_list
max_burden = True

no_variant_mask = region_genotypes.sum(axis = 0) == 0

var_scores = region_annotations[annotation_list].fill_nan(0).to_numpy().astype(np.float32).transpose()  

gis_max_list = []
gis_top2_sum_list = []
for a in tqdm(range(var_scores.shape[0])):
    burden = np.abs(np.expand_dims(var_scores[a, :], axis=1) * region_genotypes)  # shape: (variants, samples)

    # Get top-k values per sample
    top2 = np.partition(burden, -2, axis=0)[-2:, :]  # shape: (k, samples)

    # Compute max (top-1) and sum of top-k
    max_vals = np.max(top2, axis=0)
    top2_sum = np.sum(top2, axis=0)

    gis_max_list.append(max_vals)
    gis_top2_sum_list.append(top2_sum)

gis_max = np.stack(gis_max_list, axis=0).transpose()    # shape: (samples, annotations)
gis_top2 = np.stack(gis_top2_sum_list, axis=0).transpose()

# Handle no-variant case
gis_max[no_variant_mask, :] = np.nan
gis_top2[no_variant_mask, :] = np.nan

In [ ]:
gis_max.shape

## Compute burdens for the small subset

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list

In [ ]:
%%time

anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'

gene_burdens_sum_df, gene_burdens_max_df, gene_burdens_top2_df, sample_id_arr, gene_id_list = get_burdens.get_burdens_array(
    anngeno_path=anngeno_path,
    associations_df_path=associations_df_path,
    maf=0.001,
    annotation_list=all_annotation_list,
    max_burden=True,
    only_snps=False,
    n_jobs=1,
    batch_size=2,
    device=device,
)


In [ ]:
gene_burdens_max_df.shape

In [ ]:
a = torch.tensor(gene_burdens_max_df[:,0,:], device=device).transpose(1,0)
a.shape

## Phenotype GIS plot

In [ ]:
all_annotation_list

In [ ]:
gene_id_list

In [ ]:
annotation = 'promoterAI' #'am_pathogenicity'
gene_id = 'ENSG00000213398' #'ENSG00000130164' # LDLR

# Get indices for the gene and annotation
gene_idx = gene_id_list.index(gene_id)
annotation_idx = all_annotation_list.index(annotation)

# Get the burden array for the specific gene and annotation
gis_df = pl.DataFrame({
    "sample": sample_id_arr,
    "gis_sum": gene_burdens_sum_df[:, gene_idx, annotation_idx],
    "gis_max": gene_burdens_max_df[:, gene_idx, annotation_idx],
    "gis_top2": gene_burdens_top2_df[:, gene_idx, annotation_idx]
})
gis_df

In [ ]:
trait = "hdl_cholesterol"
# pheno_df = pl.read_parquet("/home/dnanexus/data_dir/phenotypes_corr/ldl_direct_prs_corrected.parquet")
pheno_df = pl.read_parquet(f"/home/dnanexus/data_dir/phenotypes_corr/{trait}_prs_corrected.parquet")
plt_df = gis_df.join(pheno_df, on="sample", how="inner")
plt_df

In [ ]:
from plotnine import *

(
    ggplot(plt_df, aes(x='gis_max', y=f'{trait}_prs_corrected')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=False) +
    labs(
        y=f'{trait} (prs corrected)'
        ) +
    theme_bw()
)

## Save Zarr

In [ ]:
def get_burdens_array_streaming(
    anngeno_path,
    genes,
    annotation_list,
    max_burden=True,
    gene_batch_size=2,
    device="cpu",
):
    """
    Generator yielding (gene, sum_burden, max_burden, top2_burden) for each gene.
    """
    from burden_utils import get_gene_burdens_numba  # assumes this is available

    gene_id_list = [g for g in genes if g in ag.region_ids]
    sample_ids = ag.samples

    for i in tqdm(range(0, len(gene_id_list), gene_batch_size)):
        batch_genes = gene_id_list[i : i + gene_batch_size]
        regions_dict = ag.get_many_regions(batch_genes)

        for gene in batch_genes:
            if gene not in regions_dict:
                continue

            region = regions_dict[gene]
            burdens = get_gene_burdens_numba(
                region["genotypes"],
                region["annotations"],
                annotation_list=annotation_list,
                max_burden=max_burden,
                device=device,
            )
            yield gene, *burdens
            del region, burdens
            gc.collect()

In [ ]:
def compute_and_store_burdens(
    config_path,
    genes,
    output_zarr,
    annotation_list,
    max_burden=True,
    gene_batch_size=2,
    device="cpu",
):
    """
    Computes and stores gene burdens directly to Zarr in streaming mode,
    with extendable gene axis.
    """
    gene_ids = [g for g in genes if g in ag.region_ids]
    sample_ids = ag.samples
    n_samples = len(sample_ids)

    # Determine number of annotations
    n_annos = len(annotation_list)

    compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)
    zarr_root = zarr.open_group(output_zarr, mode="a")

    # Create or extend datasets
    if "sum_burdens" in zarr_root:
        sum_burdens = zarr_root["sum_burdens"]
        max_burdens = zarr_root["max_burdens"]
        top2_burdens = zarr_root["top2_burdens"]
        gene_offset = sum_burdens.shape[1]
        sum_burdens.resize(n_samples, gene_offset + len(gene_ids), n_annos)
        max_burdens.resize(n_samples, gene_offset + len(gene_ids), n_annos)
        top2_burdens.resize(n_samples, gene_offset + len(gene_ids), n_annos)
    else:
        sum_burdens = zarr_root.create_dataset(
            "sum_burdens",
            shape=(n_samples, 0, n_annos),
            maxshape=(n_samples, None, n_annos),
            chunks=(n_samples, 1, 1),
            dtype="f4",
            compressor=compressor
        )
        max_burdens = zarr_root.create_dataset(
            "max_burdens",
            shape=(n_samples, 0, n_annos),
            maxshape=(n_samples, None, n_annos),
            chunks=(n_samples, 1, 1),
            dtype="f4",
            compressor=compressor
        )
        top2_burdens = zarr_root.create_dataset(
            "top2_burdens",
            shape=(n_samples, 0, n_annos),
            maxshape=(n_samples, None, n_annos),
            chunks=(n_samples, 1, 1),
            dtype="f4",
            compressor=compressor
        )
        zarr_root.create_dataset("samples", data=sample_ids, dtype="U50")
        zarr_root.create_dataset("annotations", data=np.array(annotation_mode), dtype="U50")
        zarr_root.create_dataset("genes", shape=(0,), maxshape=(None,), chunks=(1,), dtype="U50")
        gene_offset = 0

    # Extend genes array
    gene_array = zarr_root["genes"]
    gene_array.resize(gene_offset + len(gene_ids))
    gene_array[gene_offset:] = gene_ids

    # Populate burdens
    gene_idx_map = {g: gene_offset + i for i, g in enumerate(gene_ids)}
    for gene, s_burden, m_burden, t2_burden in get_burdens_array_streaming(
        ag, gene_ids, annotation_mode, gene_batch_size, device
    ):
        idx = gene_idx_map[gene]
        sum_burdens[:, idx, :] = s_burden
        max_burdens[:, idx, :] = m_burden
        top2_burdens[:, idx, :] = t2_burden
        gc.collect()


## Save all correlations